### Notebook to render evaluated predator-prey trajectories


In [43]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import HTML, Image, Video, display
except ImportError:
    HTML = Image = Video = None

    def display(value):
        print(value)
from matplotlib import animation
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle

# Pick the evaluation output directory and the evaluated world to render.
# Use data/random/training2 for the random-agent evaluation output.
RENDER_DATA_PATH = Path("data/random/training2")
EVAL_STEPS = 15000  # Set to None to auto-detect the latest *_sheep_xs.npy prefix.
TRAJECTORY_INDEX = 5

WORLD_SIZE_X = 500.0
WORLD_SIZE_Y = 500.0
GRASS_RADIUS = 200.0 #1000.0
SHEEP_RADIUS = 5.0
WOLF_RADIUS = 5.0

FRAME_STRIDE = 1
FPS = 10
START_FRAME = 0
END_FRAME = 1500

if animation.writers.is_available("ffmpeg"):
    VIDEO_WRITER = "ffmpeg"
    OUTPUT_VIDEO = Path(f"agent_movement_idx{TRAJECTORY_INDEX}.mp4")
elif animation.writers.is_available("pillow"):
    VIDEO_WRITER = "pillow"
    OUTPUT_VIDEO = Path(f"agent_movement_idx{TRAJECTORY_INDEX}.gif")
else:
    VIDEO_WRITER = None
    OUTPUT_VIDEO = None
print(f"Using matplotlib writer: {VIDEO_WRITER or 'jshtml'}")


In [44]:
RENDER_KEYS = (
    "sheep_xs",
    "sheep_ys",
    "sheep_angles",
    "wolf_xs",
    "wolf_ys",
    "wolf_angles",
)


def infer_eval_steps(data_path):
    prefixes = []
    for file_path in data_path.glob("*_sheep_xs.npy"):
        prefix = file_path.name.split("_", 1)[0]
        if prefix.isdigit():
            prefixes.append(int(prefix))
    if not prefixes:
        raise FileNotFoundError(
            f"No evaluation render files found in {data_path}. "
            "Run test_evolved.py or test_rand.py after the render-data save patch."
        )
    return max(prefixes)


def load_eval_render_data(data_path, eval_steps=None):
    data_path = Path(data_path)
    if eval_steps is None:
        eval_steps = infer_eval_steps(data_path)

    data = {}
    missing = []
    for key in RENDER_KEYS:
        file_path = data_path / f"{eval_steps}_{key}.npy"
        if file_path.exists():
            data[key] = np.load(file_path, mmap_mode="r")
        else:
            missing.append(file_path)

    if missing:
        missing_text = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing render trajectory files:\n{missing_text}")

    encounter_path = data_path / f"{eval_steps}_sheep_wolf_encounter_trajs.npy"
    encounter_totals_path = data_path / f"{eval_steps}_sheep_wolf_encounter_totals.npy"
    if encounter_path.exists():
        data["sheep_wolf_encounters"] = np.load(encounter_path, mmap_mode="r")
    if encounter_totals_path.exists():
        data["sheep_wolf_encounter_totals"] = np.load(encounter_totals_path, mmap_mode="r")

    return eval_steps, data


eval_steps, data = load_eval_render_data(RENDER_DATA_PATH, EVAL_STEPS)
print(f"Loaded render data from {RENDER_DATA_PATH} with eval_steps={eval_steps}")
print(*(f"{key}: {value.shape}" for key, value in data.items()), sep="\n")


In [45]:
def drop_trailing_singleton(array):
    if array.ndim > 0 and array.shape[-1] == 1:
        return array[..., 0]
    return array


def select_trajectory(data, trajectory_index):
    trajectory = {}
    for key in RENDER_KEYS:
        array = drop_trailing_singleton(data[key])

        if array.ndim == 3:
            if trajectory_index < 0 or trajectory_index >= array.shape[0]:
                raise IndexError(
                    f"TRAJECTORY_INDEX={trajectory_index} is out of range for {key}; "
                    f"valid range is 0..{array.shape[0] - 1}."
                )
            trajectory[key] = np.asarray(array[trajectory_index])
        elif array.ndim == 2:
            if trajectory_index not in (0, -1):
                raise IndexError(
                    f"{key} contains one trajectory only, so use TRAJECTORY_INDEX=0."
                )
            trajectory[key] = np.asarray(array)
        else:
            raise ValueError(f"Expected {key} to have 2 or 3 dimensions after squeeze, got {array.shape}")

    if "sheep_wolf_encounters" in data:
        encounter_array = drop_trailing_singleton(data["sheep_wolf_encounters"])
        trajectory["sheep_wolf_encounters"] = np.asarray(
            encounter_array[trajectory_index] if encounter_array.ndim == 2 else encounter_array
        )
    if "sheep_wolf_encounter_totals" in data:
        total_array = data["sheep_wolf_encounter_totals"]
        trajectory["sheep_wolf_encounter_total"] = float(
            total_array[trajectory_index] if total_array.ndim == 1 else total_array
        )

    return trajectory


episode_data = select_trajectory(data, TRAJECTORY_INDEX)
print(f"Selected trajectory index {TRAJECTORY_INDEX}")
print(*(f"{key}: {getattr(value, 'shape', value)}" for key, value in episode_data.items()), sep="\n")


In [46]:
def arena_mask(xs, ys):
    return (
        np.isfinite(xs)
        & np.isfinite(ys)
        & (np.abs(xs) <= WORLD_SIZE_X)
        & (np.abs(ys) <= WORLD_SIZE_Y)
    )


def masked_offsets(xs, ys):
    mask = arena_mask(xs, ys)
    return np.column_stack((np.where(mask, xs, np.nan), np.where(mask, ys, np.nan)))


def render_one_traj(render_data, output_path, frame_stride=1, fps=20):
    sheep_xs = render_data["sheep_xs"]
    sheep_ys = render_data["sheep_ys"]
    wolf_xs = render_data["wolf_xs"]
    wolf_ys = render_data["wolf_ys"]
    encounter_traj = render_data.get("sheep_wolf_encounters")

    frames = np.arange(START_FRAME, min(END_FRAME, sheep_xs.shape[0]), frame_stride)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-WORLD_SIZE_X, WORLD_SIZE_X)
    ax.set_ylim(-WORLD_SIZE_Y, WORLD_SIZE_Y)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Evaluation trajectory {TRAJECTORY_INDEX}")

    if GRASS_RADIUS is not None and GRASS_RADIUS > 0:
        ax.add_patch(
            Circle(
                (0, 0),
                GRASS_RADIUS,
                linewidth=1.5,
                edgecolor="forestgreen",
                facecolor="forestgreen",
                alpha=0.12,
            )
        )

    frame0 = frames[0]
    sheep_scatter = ax.scatter(
        sheep_xs[frame0], sheep_ys[frame0], c="royalblue", s=SHEEP_RADIUS * 4, label="Sheep"
    )
    wolf_scatter = ax.scatter(
        wolf_xs[frame0], wolf_ys[frame0], c="firebrick", s=WOLF_RADIUS * 5, label="Wolves"
    )

    frame_label = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top")
    ax.legend(loc="upper right")

    def update(frame):
        sheep_scatter.set_offsets(masked_offsets(sheep_xs[frame], sheep_ys[frame]))
        wolf_scatter.set_offsets(masked_offsets(wolf_xs[frame], wolf_ys[frame]))

        if encounter_traj is None:
            frame_label.set_text(f"t = {frame}")
        else:
            frame_label.set_text(f"t = {frame} | encounters = {encounter_traj[frame]:.0f}")
        return sheep_scatter, wolf_scatter, frame_label

    ani = FuncAnimation(fig, update, frames=frames, blit=False)
    if VIDEO_WRITER is None:
        html = ani.to_jshtml(fps=fps)
        plt.close(fig)
        return None, html

    ani.save(output_path, writer=VIDEO_WRITER, fps=fps)
    plt.close(fig)
    return output_path, None

In [47]:
video_path, video_html = render_one_traj(episode_data, OUTPUT_VIDEO, frame_stride=FRAME_STRIDE, fps=FPS)
if video_path is None:
    display(HTML(video_html) if HTML is not None else "Rendered jshtml animation")
else:
    print(f"Saved {video_path}")
    if video_path.suffix == ".mp4" and Video is not None:
        display(Video(str(video_path), embed=True))
    elif video_path.suffix == ".gif" and Image is not None:
        display(Image(filename=str(video_path)))


In [48]:
# save screenshot
import os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

def save_svg_frames(render_data, output_dir, frames_to_save):
    """
    Renders specific frames of the trajectory and saves them as SVG files.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    sheep_xs = render_data["sheep_xs"]
    sheep_ys = render_data["sheep_ys"]
    wolf_xs = render_data["wolf_xs"]
    wolf_ys = render_data["wolf_ys"]
    encounter_traj = render_data.get("sheep_wolf_encounters")

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-WORLD_SIZE_X, WORLD_SIZE_X)
    ax.set_ylim(-WORLD_SIZE_Y, WORLD_SIZE_Y)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Evaluation trajectory {TRAJECTORY_INDEX}")

    # Draw the grass area
    if GRASS_RADIUS is not None and GRASS_RADIUS > 0:
        ax.add_patch(
            Circle(
                (0, 0),
                GRASS_RADIUS,
                linewidth=1.5,
                edgecolor="forestgreen",
                facecolor="forestgreen",
                alpha=0.12,
            )
        )

    # Initialize scatters and text
    sheep_scatter = ax.scatter([], [], c="royalblue", s=SHEEP_RADIUS * 4, label="Sheep")
    wolf_scatter = ax.scatter([], [], c="firebrick", s=WOLF_RADIUS * 5, label="Wolves")
    frame_label = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top")
    ax.legend(loc="upper right")

    # Loop through the desired frames, update positions, and save
    for frame in frames_to_save:
        # Check to ensure frame is within bounds
        if frame >= sheep_xs.shape[0]:
            print(f"Skipping frame {frame} (out of bounds).")
            continue

        sheep_scatter.set_offsets(masked_offsets(sheep_xs[frame], sheep_ys[frame]))
        wolf_scatter.set_offsets(masked_offsets(wolf_xs[frame], wolf_ys[frame]))

        frame_label.set_text(f"t = {frame}")

        # Save to SVG
        out_path = output_dir / f"frame_{frame:04d}.svg"
        fig.savefig(out_path, format="svg", bbox_inches="tight")
        print(f"Saved {out_path}")

    plt.close(fig)

# --- Example Usage ---
# Create a list of frames you want to export (e.g., every 100th frame from START_FRAME to END_FRAME)
frames_to_export = range(START_FRAME, min(END_FRAME, episode_data["sheep_xs"].shape[0]), 100)

# Call the function (creates a folder named 'svg_screenshots' and saves SVGs there)
save_svg_frames(episode_data, output_dir="figures/random_snapshots", frames_to_save=frames_to_export)